> **Original course-project notebook (2026), kept for reference; superseded by `src/dfer/`.** Outputs are cleared (they contained DFEW frames, which cannot be redistributed). See the README for the final pipeline and results.

In [ ]:
# ==========================================
# 1) Setup & Load Models Manually
# ==========================================
import os
import torch
import cv2
import numpy as np
import urllib.request
import torch.nn.functional as F
from transformers import VideoMAEConfig, VideoMAEForVideoClassification
from transformers import VJEPA2Config, VJEPA2ForVideoClassification
from torchvision import transforms

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Define the paths to your models
model_paths = {
    'ViT Baseline': r"data/outputs\checkpoints\02_supervised_vit_baseline_fold1_FINAL_best_uar.pth",
    'MAE SSL': r"data/outputs\checkpoints\mae_ssl_fold1_FINAL_best_uar.pth",
    'V-JEPA': r"data/outputs\checkpoints\jepa_fold1_FINAL_stage2_partial_best_uar.pth",
    'MAE-DFER SSL': r"data/outputs\checkpoints\mae_dfer_fold1_FINAL_stage2_partial_best_uar.pth"
}

def load_model(model_path, model_class, num_labels, is_vjepa=False):
    if is_vjepa:
        config = VJEPA2Config.from_pretrained("facebook/vjepa2-vitl-fpc64-256", num_labels=num_labels)
    else:
        config = VideoMAEConfig(num_labels=num_labels)
    
    model = model_class(config) 
    checkpoint = torch.load(model_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    
    # Standard fix for VideoMAE key mismatches
    if 'videomae.layernorm.weight' in state_dict:
        state_dict['fc_norm.weight'] = state_dict.pop('videomae.layernorm.weight')
    if 'videomae.layernorm.bias' in state_dict:
        state_dict['fc_norm.bias'] = state_dict.pop('videomae.layernorm.bias')
    
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

# Load all models
models = {
    'ViT Baseline': load_model(model_paths['ViT Baseline'], VideoMAEForVideoClassification, 7),
    'MAE SSL': load_model(model_paths['MAE SSL'], VideoMAEForVideoClassification, 7),
    'V-JEPA': load_model(model_paths['V-JEPA'], VJEPA2ForVideoClassification, 7, is_vjepa=True),
    'MAE-DFER SSL': load_model(model_paths['MAE-DFER SSL'], VideoMAEForVideoClassification, 7)
}
print("All models loaded successfully!")

In [ ]:
# ========================================================================
# 2) Preprocessing (Square Face Cropping + Uniform Sampling)
# ========================================================================

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

cascade_path = 'haarcascade_frontalface_default.xml'
if not os.path.exists(cascade_path):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml', cascade_path)

face_cascade = cv2.CascadeClassifier(cascade_path)

def preprocess_video(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int) if total_frames >= num_frames else np.arange(total_frames)
    
    frames, last_face_box = [], None
    current_frame = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if current_frame in indices:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(100, 100))
            if len(faces) > 0:
                x, y, w, h = faces[0]
                size = max(w, h)
                pad = int(size * 0.3)
                half_size = (size // 2) + pad
                cx, cy = x + w // 2, y + h // 2
                y1, y2 = max(0, cy - half_size), min(frame.shape[0], cy + half_size)
                x1, x2 = max(0, cx - half_size), min(frame.shape[1], cx + half_size)
                last_face_box = (x1, y1, x2, y2)
                frame = frame[y1:y2, x1:x2]
            elif last_face_box is not None:
                x1, y1, x2, y2 = last_face_box
                frame = frame[y1:y2, x1:x2]
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(transform(frame))
        current_frame += 1
        if len(frames) == num_frames: break
    cap.release()
    
    if len(frames) < num_frames:
        frames.extend([frames[-1]] * (num_frames - len(frames)))
    return torch.stack(frames[:num_frames]).unsqueeze(0).to(device)

In [ ]:
# ========================================================================
# 3) Run Inference & Print Probabilities for Cross-Author Experiment
# ========================================================================

emotion_classes = ["Happy", "Sad", "Neutral", "Angry", "Surprise", "Disgust", "Fear"]
base_dir = r"data/Testing Videos"

def get_predictions(model, video_tensor):
    with torch.no_grad():
        if "VJEPA" in model.__class__.__name__:
            outputs = model(pixel_values_videos=video_tensor)
        else:
            outputs = model(pixel_values=video_tensor)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1).cpu().numpy()[0] 
        pred_idx = logits.argmax(dim=1).cpu().numpy()[0]
    return pred_idx, probs

# Group videos by Subject (Both using Normal/Natural Expressions)
video_sets = {
    "Set A (Subject A - Female)": [
        os.path.join(base_dir, "Happy (1).MOV"),
        os.path.join(base_dir, "Sad (1).MOV"),
        os.path.join(base_dir, "Neutral (1).MOV"),
        os.path.join(base_dir, "Angry (1).MOV"),
        os.path.join(base_dir, "Surprised (1).MOV")
    ],
    "Set B (Subject B - Male)": [
        os.path.join(base_dir, "Happy (2).MOV"),
        os.path.join(base_dir, "Sad (2).MOV"),
        os.path.join(base_dir, "Neutral (2).MOV"),
        os.path.join(base_dir, "Angry (2).MOV"),
        os.path.join(base_dir, "Surprised (2).MOV")
    ]
}

# Nested dictionary to store all probabilities for the final heatmap
all_prob_results = {}

for set_name, v_paths in video_sets.items():
    print(f"\n{'='*70}\nTarget: {set_name}\n{'='*70}")
    set_probs = {}
    
    for model_name, model in models.items():
        print(f"\nTesting Architecture: {model_name}")
        print("-" * 50)
        model_set_probs = {}
        
        for video_path in v_paths:
            if not os.path.exists(video_path):
                continue
            
            v_name = os.path.basename(video_path)
            video_tensor = preprocess_video(video_path)
            pred_idx, probs = get_predictions(model, video_tensor)
            
            # Save raw probabilities for the heatmap
            model_set_probs[v_name] = probs
            
            predicted_emotion = emotion_classes[pred_idx]
            probs_str = " | ".join([f"{emotion_classes[j]}: {p*100:.1f}%" for j, p in enumerate(probs)])
            
            print(f"Video: {v_name}")
            print(f" -> Result: {predicted_emotion}")
            print(f" -> Confidences: {probs_str}\n")
            
        set_probs[model_name] = model_set_probs
    all_prob_results[set_name] = set_probs

In [ ]:
# ========================================================================
# 4) Visualize Full Confidence Distributions (Probability Profiles)
# ========================================================================
import matplotlib.pyplot as plt
import numpy as np

# Ground Truth Confidence Data
# This captures exactly what % the model gave to the CORRECT emotion.
models = ["ViT Baseline", "MAE SSL", "V-JEPA", "MAE-DFER SSL"]
emotions = ["Happy", "Sad", "Neutral", "Angry", "Surprised"]

# Data for Subject A (Female)
# [ViT, MAE, V-JEPA, MAE-DFER]
data_a = np.array([
    [79.7, 55.1, 68.5, 16.1],  # Happy GT
    [70.5, 62.6, 52.1, 36.9],  # Sad GT
    [21.9, 10.9, 6.8, 25.9],   # Neutral GT
    [3.8, 2.1, 0.2, 7.9],      # Angry GT
    [8.4, 7.0, 17.6, 7.9]      # Surprised GT
])

# Data for Subject B (Male) - From your latest logs
data_b = np.array([
    [60.9, 14.7, 0.2, 1.9],    # Happy GT
    [24.3, 23.8, 32.9, 56.4],  # Sad GT
    [37.2, 9.9, 20.2, 5.7],    # Neutral GT
    [11.0, 6.6, 0.1, 1.9],     # Angry GT
    [3.7, 13.1, 17.3, 14.0]    # Surprised GT
])

def plot_heatmap(data, title):
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(data, cmap="Blues", vmin=0, vmax=100)
    
    # Add Colorbar
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("Confidence in Correct Emotion (%)", rotation=-90, va="bottom")

    # Set Labels
    ax.set_xticks(np.arange(len(models)))
    ax.set_xticklabels(models)
    ax.set_yticks(np.arange(len(emotions)))
    ax.set_yticklabels(emotions)

    # Annotate values inside cells
    for i in range(len(emotions)):
        for j in range(len(models)):
            ax.text(j, i, f"{data[i, j]:.1f}%", ha="center", va="center", 
                    color="white" if data[i, j] > 50 else "black", fontweight='bold')

    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

# Run for both subjects
plot_heatmap(data_a, "Inference Success: Subject A (Female)")
plot_heatmap(data_b, "Inference Success: Subject B (Male)")